<a href="https://colab.research.google.com/github/Navaneethan2398/Simple_RAG_Project/blob/main/Simple_RAG_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q \
langchain==0.3.26 \
langchain-core==0.3.68 \
langchain-community==0.3.27 \
langchain-text-splitters==0.3.8 \
langchain-google-genai==2.1.5

In [ ]:
!pip install -q unstructured

In [ ]:
from langchain_community.document_loaders import UnstructuredURLLoader

urls = ['https://learn.microsoft.com/en-us/intune/configmgr']
loader = UnstructuredURLLoader(urls=urls)
data = loader.load()

In [ ]:
data

[Document(metadata={'source': 'https://learn.microsoft.com/en-us/intune/configmgr'}, page_content="Microsoft Configuration Manager documentation\n\nOfficial product documentation for Configuration Manager\n\n\n\nTraining\n\nWhat's new\n\n\n\nArchitecture\n\nSupported configurations\n\n\n\nDeploy\n\nUpdates and servicing\n\n\n\nTraining\n\nLog files\n\n\n\nTraining\n\nPorts\n\n\n\nGet started\n\nTechnical preview\n\nCloud-attached management\n\nOverview of cloud attach\n\nMicrosoft Intune tenant attach\n\nEndpoint analytics\n\nOverview of cloud management gateway (CMG)\n\nPlan for Microsoft Entra ID\n\nCo-management\n\nQuickstarts\n\nTutorial\n\nWorkloads\n\nSee more\n\nReal-time management\n\nCMPivot\n\nRun PowerShell scripts\n\nInfrastructure simplification\n\nSite server high availability\n\nReassign a distribution point\n\nConfiguration Manager on Azure\n\nCore infrastructure\n\nUsing the console\n\nDeploy clients\n\nStay current with ConfigMgr and Windows 10\n\nSee more\n\nApp mana

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# split data
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500)
docs = text_splitter.split_documents(data)


print("Total number of documents: ",len(docs))

Total number of documents:  8


In [ ]:
docs[0]

Document(metadata={'source': 'https://learn.microsoft.com/en-us/intune/configmgr'}, page_content="Microsoft Configuration Manager documentation\n\nOfficial product documentation for Configuration Manager\n\n\n\nTraining\n\nWhat's new\n\n\n\nArchitecture\n\nSupported configurations\n\n\n\nDeploy\n\nUpdates and servicing\n\n\n\nTraining\n\nLog files\n\n\n\nTraining\n\nPorts\n\n\n\nGet started\n\nTechnical preview\n\nCloud-attached management\n\nOverview of cloud attach\n\nMicrosoft Intune tenant attach\n\nEndpoint analytics\n\nOverview of cloud management gateway (CMG)\n\nPlan for Microsoft Entra ID\n\nCo-management\n\nQuickstarts\n\nTutorial")

In [ ]:
!pip install -q chromadb langchain-community

In [ ]:
import os

os.environ["GOOGLE_API_KEY"] = "GOOGLE_API_KEY"

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

vector = embeddings.embed_query("Hello world")
print(vector[:5])

[-0.023421520367264748, 0.016765719279646873, 0.009261323139071465, -0.06383000314235687, -0.00262627680785954]


In [ ]:
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings
)

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":2})

In [ ]:
retrieved_docs = retriever.invoke("What kind of services they provide?")

In [ ]:
len(retrieved_docs)

2

In [ ]:
retrieved_docs

[Document(metadata={'source': 'https://learn.microsoft.com/en-us/intune/configmgr'}, page_content='Product support\n\nGet professional help from Microsoft support\n\nSystem Center 2012 R2 Configuration Manager\n\nDocumentation for the previous version of Configuration Manager\n\nOther content\n\nDevelop\n\nPowerShell\n\nSoftware development kit (SDK)\n\nAdministration service REST API\n\nSQL views\n\nTools\n\nSupport Center\n\nMDT\n\nPackage Conversion Manager\n\nSee more >\n\nOther community sites\n\nConfigMgr on Reddit\n\nConfigMgr Professionals Group on Facebook'),
 Document(metadata={'source': 'https://learn.microsoft.com/en-us/intune/configmgr'}, page_content='Product support\n\nGet professional help from Microsoft support\n\nSystem Center 2012 R2 Configuration Manager\n\nDocumentation for the previous version of Configuration Manager\n\nOther content\n\nDevelop\n\nPowerShell\n\nSoftware development kit (SDK)\n\nAdministration service REST API\n\nSQL views\n\nTools\n\nSupport Cent

In [ ]:
print(retrieved_docs[0].page_content)

Product support

Get professional help from Microsoft support

System Center 2012 R2 Configuration Manager

Documentation for the previous version of Configuration Manager

Other content

Develop

PowerShell

Software development kit (SDK)

Administration service REST API

SQL views

Tools

Support Center

MDT

Package Conversion Manager

See more >

Other community sites

ConfigMgr on Reddit

ConfigMgr Professionals Group on Facebook


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.4,
    max_output_tokens=200
)

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [ ]:
response = rag_chain.invoke({"input": "What is cloud attach"})
print(response["answer"])

Cloud attach is a feature that connects Microsoft Configuration Manager to Microsoft Intune. This integration allows for unified management of devices, enabling features like tenant attach and co-management. It simplifies infrastructure and provides real-time management capabilities.
